# AI-Driven Healthcare Decision Support System

This project implements a healthcare decision support system that uses
AI and a medical knowledge base to safely analyze user symptoms.

Key principles:
- The system is NOT a chatbot, but a healthcare platform
- Only OTC medications are suggested
- Prescription or harmful drugs are blocked
- High-risk cases trigger real doctor intervention
- AI responses are grounded in verified medical knowledge using a vector database

In [3]:
# Medical knowledge base (verified & safe content only)

medical_documents = [
    """
    Condition: Fever
    Symptoms: high body temperature, body pain, headache
    Description: Fever is a temporary increase in body temperature, often due to infection.
    OTC Medication: Paracetamol
    Safety Notes: Avoid overdose. Consult a doctor if fever lasts more than 3 days.
    """,

    """
    Condition: Headache
    Symptoms: head pain, pressure, mild dizziness
    Description: Headache may occur due to stress, dehydration, or lack of sleep.
    OTC Medication: Paracetamol
    Safety Notes: Seek medical attention if severe or persistent.
    """,

    """
    Condition: Acidity
    Symptoms: stomach burning, acid reflux, indigestion
    Description: Acidity occurs due to excess stomach acid.
    OTC Medication: Antacids
    Safety Notes: Avoid spicy food. Consult a doctor if frequent.
    """,

    """
    Condition: Common Cold
    Symptoms: runny nose, sneezing, sore throat
    Description: Common cold is a viral infection affecting the nose and throat.
    OTC Medication: Antihistamines, Paracetamol
    Safety Notes: Rest and hydration recommended.
    """
]

print(f"Loaded {len(medical_documents)} medical knowledge entries.")

Loaded 4 medical knowledge entries.


In [28]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

In [5]:
# Generate embeddings for medical documents
medical_embeddings = embedding_model.encode(medical_documents)

medical_embeddings.shape

(4, 384)

In [6]:
import faiss
import numpy as np

# Convert embeddings to numpy array
embeddings_np = np.array(medical_embeddings).astype("float32")

# Create FAISS index
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embeddings_np)

print("Vector database created successfully.")

Vector database created successfully.


In [7]:
def search_medical_knowledge(query, top_k=2):
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(medical_documents[idx])

    return results

In [8]:
query = "I have fever and body pain"
results = search_medical_knowledge(query)

for res in results:
    print("---- Retrieved Medical Knowledge ----")
    print(res)

---- Retrieved Medical Knowledge ----

    Condition: Fever
    Symptoms: high body temperature, body pain, headache
    Description: Fever is a temporary increase in body temperature, often due to infection.
    OTC Medication: Paracetamol
    Safety Notes: Avoid overdose. Consult a doctor if fever lasts more than 3 days.
    
---- Retrieved Medical Knowledge ----

    Condition: Common Cold
    Symptoms: runny nose, sneezing, sore throat
    Description: Common cold is a viral infection affecting the nose and throat.
    OTC Medication: Antihistamines, Paracetamol
    Safety Notes: Rest and hydration recommended.
    


In [9]:
# -------------------------------
# Medication Safety Configuration
# -------------------------------

# Allowed OTC medications (safe list)
OTC_MEDICATIONS = {
    "paracetamol",
    "antacids",
    "ors",
    "cetirizine"
}

# Prescription / dangerous medication keywords
BLOCKED_MEDICATION_KEYWORDS = {
    "sleeping pill",
    "sleeping pills",
    "antibiotic",
    "antibiotics",
    "painkiller",
    "opioid",
    "steroid",
    "antidepressant",
    "antipsychotic",
    "sedative",
    "benzodiazepine"
}

# High-risk symptom keywords (auto doctor)
HIGH_RISK_SYMPTOMS = {
    "chest pain",
    "shortness of breath",
    "breathlessness",
    "seizure",
    "unconscious",
    "pregnancy",
    "suicidal",
    "severe bleeding"
}

In [10]:
def medication_safety_check(user_input: str, retrieved_docs: list):
    text = user_input.lower()

    # 1. High-risk symptom detection
    for symptom in HIGH_RISK_SYMPTOMS:
        if symptom in text:
            return {
                "status": "ESCALATE",
                "reason": "High-risk symptoms detected",
                "action": "Doctor intervention required"
            }

    # 2. Blocked medication intent detection
    for keyword in BLOCKED_MEDICATION_KEYWORDS:
        if keyword in text:
            return {
                "status": "ESCALATE",
                "reason": "Prescription or dangerous medication detected",
                "action": "Doctor intervention required"
            }

    # 3. Extract suggested OTC meds from retrieved docs
    safe_meds = []
    for doc in retrieved_docs:
        for med in OTC_MEDICATIONS:
            if med in doc.lower():
                safe_meds.append(med)

    if safe_meds:
        return {
            "status": "SAFE",
            "medications": list(set(safe_meds)),
            "action": "OTC medication guidance allowed"
        }

    # 4. Default safe fallback
    return {
        "status": "SAFE",
        "medications": [],
        "action": "General advice only"
    }

## Checking Status safe or escalate

In [11]:
user_query = "I have fever and headache"
docs = search_medical_knowledge(user_query)

medication_safety_check(user_query, docs)

{'status': 'SAFE',
 'medications': ['paracetamol'],
 'action': 'OTC medication guidance allowed'}

In [12]:
user_query = "I cannot sleep properly, can you suggest sleeping pills?"
docs = search_medical_knowledge(user_query)

medication_safety_check(user_query, docs)

{'status': 'ESCALATE',
 'reason': 'Prescription or dangerous medication detected',
 'action': 'Doctor intervention required'}

In [13]:
user_query = "I have chest pain and difficulty breathing"
docs = search_medical_knowledge(user_query)

medication_safety_check(user_query, docs)

{'status': 'ESCALATE',
 'reason': 'High-risk symptoms detected',
 'action': 'Doctor intervention required'}

In [14]:
def doctor_escalation_handler(reason):
    return {
        "message": "This condition requires professional medical supervision.",
        "next_step": "Connecting you to a certified doctor",
        "reason": reason
    }

In [15]:
def healthcare_system_response(user_query):
    retrieved_docs = search_medical_knowledge(user_query)
    safety_result = medication_safety_check(user_query, retrieved_docs)

    if safety_result["status"] == "ESCALATE":
        return doctor_escalation_handler(safety_result["reason"])

    return {
        "message": "Based on your symptoms, here is some safe guidance.",
        "medications": safety_result["medications"],
        "note": "Consult a doctor if symptoms persist or worsen."
    }

In [16]:
healthcare_system_response("I have fever and body pain")

{'message': 'Based on your symptoms, here is some safe guidance.',
 'medications': ['paracetamol'],
 'note': 'Consult a doctor if symptoms persist or worsen.'}

In [17]:
healthcare_system_response("I want sleeping pills for better sleep")

{'message': 'This condition requires professional medical supervision.',
 'next_step': 'Connecting you to a certified doctor',
 'reason': 'Prescription or dangerous medication detected'}

In [18]:
# -------------------------------
# Risk Classification Configuration
# -------------------------------

MILD_SYMPTOMS = {
    "fever",
    "headache",
    "cold",
    "cough",
    "acidity",
    "body pain",
    "runny nose"
}

MODERATE_SYMPTOMS = {
    "persistent fever",
    "vomiting",
    "severe headache",
    "diarrhea",
    "dizziness"
}

SEVERE_SYMPTOMS = {
    "chest pain",
    "shortness of breath",
    "breathlessness",
    "seizure",
    "unconscious",
    "severe bleeding",
    "pregnancy"
}

In [19]:
def classify_risk(user_input: str):
    text = user_input.lower()

    for symptom in SEVERE_SYMPTOMS:
        if symptom in text:
            return "SEVERE"

    for symptom in MODERATE_SYMPTOMS:
        if symptom in text:
            return "MODERATE"

    for symptom in MILD_SYMPTOMS:
        if symptom in text:
            return "MILD"

    # Default fallback
    return "UNKNOWN"

In [20]:
print(classify_risk("I have fever and headache"))
print(classify_risk("I have vomiting and dizziness"))
print(classify_risk("I have chest pain"))

MILD
MODERATE
SEVERE


In [21]:
def classify_risk(user_input: str):
    text = user_input.lower()

    for symptom in SEVERE_SYMPTOMS:
        if symptom in text:
            return "SEVERE"

    for symptom in MODERATE_SYMPTOMS:
        if symptom in text:
            return "MODERATE"

    for symptom in MILD_SYMPTOMS:
        if symptom in text:
            return "MILD"

    # Default fallback
    return "UNKNOWN"

In [22]:
def build_llm_context(retrieved_docs):
    context = "\n".join(retrieved_docs)
    return f"""
You are a healthcare assistant.
Use ONLY the medical information provided below.
Do NOT suggest prescription drugs.
If unsure, advise consulting a doctor.

Medical Information:
{context}
"""

In [23]:
def simulated_llm_response(context, user_query):
    return (
        "Based on the provided medical information, your symptoms appear to be "
        "commonly associated with minor conditions. General guidance and safe "
        "OTC options may help. Please consult a doctor if symptoms persist."
    )

In [24]:
def ai_healthcare_pipeline(user_query):
    # Step 1: Risk classification
    risk = classify_risk(user_query)

    if risk == "SEVERE":
        return doctor_escalation_handler("Severe risk detected")

    # Step 2: Retrieve medical knowledge
    retrieved_docs = search_medical_knowledge(user_query)

    # Step 3: Medication safety check
    safety_result = medication_safety_check(user_query, retrieved_docs)

    if safety_result["status"] == "ESCALATE":
        return doctor_escalation_handler(safety_result["reason"])

    # Step 4: Build LLM context
    context = build_llm_context(retrieved_docs)

    # Step 5: LLM response (simulated)
    llm_text = simulated_llm_response(context, user_query)

    return {
        "risk_level": risk,
        "response": llm_text,
        "safe_medications": safety_result.get("medications", []),
        "note": "This is not a diagnosis. Consult a doctor if symptoms persist."
    }

In [25]:
ai_healthcare_pipeline("I have fever and body pain")

{'risk_level': 'MILD',
 'response': 'Based on the provided medical information, your symptoms appear to be commonly associated with minor conditions. General guidance and safe OTC options may help. Please consult a doctor if symptoms persist.',
 'safe_medications': ['paracetamol'],
 'note': 'This is not a diagnosis. Consult a doctor if symptoms persist.'}

## LLM connection (ollama)

In [26]:
!pip install ollama

In [27]:
import ollama

def local_llm_response(context, user_query):

    prompt = f"""
You are a healthcare assistant.

Rules:
- Use only the provided medical knowledge.
- Do not suggest prescription medication.
- Recommend consulting a doctor when uncertain.

Medical Knowledge:
{context}

User Query:
{user_query}
"""

    response = ollama.chat(
        model="phi3",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]

In [29]:
def ai_healthcare_pipeline(user_query):

    risk = classify_risk(user_query)

    if risk == "SEVERE":
        return doctor_escalation_handler("Severe risk detected")

    retrieved_docs = search_medical_knowledge(user_query)

    safety_result = medication_safety_check(user_query, retrieved_docs)

    if safety_result["status"] == "ESCALATE":
        return doctor_escalation_handler(safety_result["reason"])

    context = build_llm_context(retrieved_docs)

    llm_text = local_llm_response(context, user_query)

    return {
        "risk_level": risk,
        "response": llm_text,
        "safe_medications": safety_result.get("medications", []),
        "note": "Consult a doctor if symptoms persist."
    }

In [31]:
ai_healthcare_pipeline("I have fever and headache")

{'risk_level': 'MILD',
 'response': "\nYou are experiencing symptoms of fever and headache. These symptoms might indicate a mild illness or be a response to stress, dehydration, or lack of sleep. It's advised to take Paracetamol for comfort, ensuring you do not exceed the recommended dose. Please monitor your symptoms, and if your fever persists for more than three days or if your headache becomes severe or persistent, it is important to consult a doctor for further evaluation and advice. Remember, it's crucial to maintain hydration and rest adequately.",
 'safe_medications': ['paracetamol'],
 'note': 'Consult a doctor if symptoms persist.'}

In [32]:
# Symptom interview question bank

SYMPTOM_QUESTIONS = {
    "fever": [
        "How long have you had the fever?",
        "Do you also have body pain or headache?",
        "Have you measured your temperature?"
    ],

    "headache": [
        "Where exactly is the headache located?",
        "How severe is the headache?",
        "Have you been under stress or lack of sleep?"
    ],

    "acidity": [
        "Do you feel burning in your stomach or chest?",
        "Did you eat spicy or oily food recently?",
        "Is the discomfort worse after meals?"
    ],

    "cold": [
        "Do you have a runny nose or sneezing?",
        "Do you have a sore throat?",
        "Do you feel body weakness?"
    ]
}

In [56]:
def detect_primary_symptom(user_input):
    
    text = user_input.lower()

    symptoms = {
        "fever": ["fever", "temperature"],
        "headache": ["headache", "head pain"],
        "acidity": ["acidity", "acid reflux", "burning stomach"],
        "cold": ["cold", "runny nose", "sneezing"],
        "body pain": ["body pain", "muscle pain"]
    }

    for symptom, keywords in symptoms.items():
        for word in keywords:
            if word in text:
                return symptom

    return None

In [34]:
def generate_interview_questions(user_query):

    symptom = detect_primary_symptom(user_query)

    if symptom is None:
        return ["Could you describe your symptoms in more detail?"]

    return SYMPTOM_QUESTIONS[symptom]

In [35]:
generate_interview_questions("I have fever")

['How long have you had the fever?',
 'Do you also have body pain or headache?',
 'Have you measured your temperature?']

In [36]:
def healthcare_interview_stage(user_query):

    questions = generate_interview_questions(user_query)

    return {
        "stage": "symptom_interview",
        "questions": questions
    }

In [37]:
healthcare_interview_stage("I have headache")

{'stage': 'symptom_interview',
 'questions': ['Where exactly is the headache located?',
  'How severe is the headache?',
  'Have you been under stress or lack of sleep?']}

## Last Stage

In [38]:
interview_state = {
    "symptom": None,
    "questions": [],
    "answers": [],
    "current_question": 0
}

In [39]:
def start_interview(user_query):

    symptom = detect_primary_symptom(user_query)

    if symptom is None:
        return "Please describe your symptoms in more detail."

    interview_state["symptom"] = symptom
    interview_state["questions"] = SYMPTOM_QUESTIONS[symptom]
    interview_state["answers"] = []
    interview_state["current_question"] = 0

    return interview_state["questions"][0]

In [40]:
start_interview("I have fever")

'How long have you had the fever?'

In [41]:
def answer_question(user_answer):

    interview_state["answers"].append(user_answer)
    interview_state["current_question"] += 1

    if interview_state["current_question"] < len(interview_state["questions"]):
        return interview_state["questions"][interview_state["current_question"]]

    return "Thank you. Analyzing your symptoms..."

In [42]:
start_interview("I have fever")

'How long have you had the fever?'

In [43]:
answer_question("2 days")

'Do you also have body pain or headache?'

In [44]:
answer_question("7 days")

'Have you measured your temperature?'

In [45]:
answer_question("No")

'Thank you. Analyzing your symptoms...'

In [47]:
query = "I have fever and headache"
ai_healthcare_pipeline(query)

{'risk_level': 'MILD',
 'response': "\nBased on the information provided:\n\nFever often indicates that the body is fighting an infection. It's accompanied by symptoms such as high body temperature, body pain, and headache. Over-the-counter (OTC) medication, like Paracetamol, is generally recommended for symptom relief. However, ensure not to exceed the recommended dosage to avoid overdose. It's also advised to seek medical attention if your fever persists for more than three days.\n\nFor the headache, which may result from stress, dehydration, or lack of sleep, Paracetamol could also be used. If your headache is severe or persistent, it's essential to consult a healthcare professional.\n\nRemember, it's always crucial to consult a doctor if you're uncertain about your symptoms or their severity.",
 'safe_medications': ['paracetamol'],
 'note': 'Consult a doctor if symptoms persist.'}

In [65]:
interview_state = {
    "symptom": None,
    "questions": [],
    "answers": [],
    "current_question": 0
}

In [57]:
def healthcare_chat(user_input):

    # Start interview if no symptom recorded yet
    if interview_state["symptom"] is None:
        return start_interview(user_input)

    # Interview in progress
    if interview_state["current_question"] < len(interview_state["questions"]):

        response = answer_question(user_input)

        # If interview finished
        if response == "Thank you. Analyzing your symptoms...":

            full_query = interview_state["symptom"] + " " + " ".join(interview_state["answers"])

            result = ai_healthcare_pipeline(full_query)

            # Reset interview
            interview_state["symptom"] = None
            interview_state["questions"] = []
            interview_state["answers"] = []
            interview_state["current_question"] = 0

            return result

        return response

    return "Please describe your symptoms."

In [58]:
healthcare_chat("I have fever")

'How long have you had the fever?'

In [59]:
healthcare_chat("5 days")

'Do you also have body pain or headache?'

In [60]:
healthcare_chat("No")

'Have you measured your temperature?'

In [61]:
healthcare_chat("yes it is 101")

{'risk_level': 'MILD',
 'response': "\nBased on the information provided, it appears that you have a fever with a body temperature of 101. Since you mentioned that the fever has lasted for 5 days, it is beyond the recommended time frame of 3 days to consult a doctor without OTC medication. I would advise seeking medical advice. However, as a healthcare assistant, I'm not able to prescribe medication. Remember that it's important to stay well-hydrated and rest as much as possible, which can also aid in your recovery.",
 'safe_medications': ['paracetamol'],
 'note': 'Consult a doctor if symptoms persist.'}

In [62]:
healthcare_chat("I am feeling low energy")

'Please describe your symptoms in more detail.'

In [67]:
healthcare_chat("body pain")

'Please describe your symptoms.'

In [68]:
healthcare_chat("Feeling pain")

'Please describe your symptoms.'